# 09 - Research and discussion
Short, simple answers. Compare them with the **numbers from your own results** (table below) and edit the text where your data disagrees.


In [1]:
import sys, warnings
sys.path.append("..")                      # lets the notebook import the reusable code in ../src
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src import config, preprocess as P, models as M
from src.evaluate import regression_metrics, permutation_table
sns.set_theme(style="whitegrid")
pd.options.display.float_format = "{:,.2f}".format
config.PLOT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Results used in this discussion

In [4]:
from src.evaluate import key_findings
test_df = pd.read_csv(config.OUT_DIR / "model_comparison.csv")
gap = pd.read_csv(config.METRIC_DIR / "overfitting_gap.csv")
focus = test_df[test_df["Model"].str.contains("Random Forest|XGBoost|LightGBM|CatBoost|Extra Trees|Ridge")]
focus = focus.merge(gap[["Model", "R2_gap"]], on="Model")
cols = [c for c in ["Model", "MAE", "RMSE", "R2", "Train_time_s", "CV_MAE_mean", "CV_MAE_std", "R2_gap"] if c in focus.columns]
focus[cols].sort_values("MAE")

,Model,MAE,RMSE,R2,Train_time_s,CV_MAE_mean,CV_MAE_std,R2_gap
1,Random Forest (Bagging),"16,346.77","35,585.50",0.85,1.77,"17,649.32",842.91,0.05
6,Stacking (Ridge meta-model),"16,861.08","34,233.64",0.86,52.11,"18,735.99",918.48,0.03
2,Extra Trees (Bagging),"17,815.98","36,291.83",0.84,2.18,"19,024.88","1,098.73",0.07
4,LightGBM (Boosting),"20,020.84","36,686.22",0.84,3.17,"21,642.78","1,204.53",-0.00
0,Ridge Regression (baseline),"20,210.51","42,353.87",0.79,0.96,"21,411.02","1,809.65",0.01
5,CatBoost (Boosting),"23,709.47","41,418.29",0.80,3.58,"25,092.78","1,798.79",-0.00
3,XGBoost (Boosting),"23,756.99","40,439.73",0.81,1.77,"24,456.78","1,738.66",0.01


In [5]:
for line in key_findings(test_df, None, gap): print("-", line)

- Best model on the untouched test set: **Hybrid (RF + ANN)** (MAE Rs. 15,810, RMSE Rs. 33,765, R² 0.865).
- Weakest model: **XGBoost (Boosting)** (MAE Rs. 23,757); the best model's MAE is 33.5% lower.
- The best ensemble cuts the MAE of the Ridge baseline by 21.8%.
- Random Forest vs gradient boosting: best booster is **LightGBM (Boosting)** (MAE 22.5% higher than Random Forest); training time RF 1.8s vs 3.2s.
- Stacking vs its best single model: MAE did not improve by 3.1%.
- Hybrid RF + ANN vs Random Forest alone: MAE 3.3% lower.
- Largest train-to-test R² gap (a sign of over-fitting): **Extra Trees (Bagging)** (0.069).


## 2. Random Forest vs gradient boosting (XGBoost, LightGBM, CatBoost)

| Aspect | Random Forest (bagging) | Gradient boosting (XGBoost / LightGBM / CatBoost) |
|---|---|---|
| How trees are built | Many deep trees **independently, in parallel**, on bootstrap samples | Many shallow trees **one after another**; each fits the previous errors |
| Main effect | Reduces **variance** (averaging de-correlated trees) | Reduces **bias** (keeps correcting mistakes); learning rate controls variance |
| Speed | Easy to parallelise; training cost grows with tree depth and number of trees | Sequential by nature, but histogram methods make LightGBM / XGBoost fast; CatBoost is usually the slowest to train on CPU |
| Tuning effort | Few important knobs; works well with defaults | More knobs (learning rate, rounds, depth, regularisation); needs tuning/early stopping to shine |
| Over-fitting risk | Low: more trees do not over-fit (error flattens) | Higher: too many rounds or too deep trees over-fit; shrinkage and regularisation are needed |
| Categorical features | scikit-learn needs encoding (one-hot here) | XGBoost needs encoding (or experimental support); LightGBM can handle categories natively; **CatBoost** handles categories natively with ordered target statistics. In this project all models use the same one-hot features for a fair comparison |
| Typical accuracy on tabular data | Strong, robust baseline | Often the best when tuned, but not always: on small data (like ~5k rows) the gap can be small or reversed |

**Which won on my data, and why?**
- Random Forest achieved the lowest MAE among the five tree models (Rs. 16,347), outperforming Extra Trees and all boosting models.
- LightGBM was the best booster but had 22% higher MAE than Random Forest.
- Random Forest also trained faster (1.8 s vs 3.2 s for LightGBM).
- The small dataset, many categorical features and noisy asking prices may have affected boosting performance.
- XGBoost was still improving at 800 rounds, suggesting that further tuning could improve results.


## 3. Stacking and hybrid
- **Stacking** helps only if the base models make *different* errors. RF and the boosters are all tree ensembles on the same features, so their errors are usually highly correlated (see notebook 05), which limits the gain.
- **Hybrid RF + ANN** mixes two very different learners (trees: piecewise-constant, good with categories and interactions; neural network: smooth function of the inputs). It can help when their errors differ, and the validation-chosen weight shows which one the data prefers.

## 4. Strengths and limitations of tree ensembles: tabular vs image/text data

**Tabular data (this project)**
- Strengths: handle mixed numeric/categorical columns, need little scaling, capture non-linear effects and interactions, work with small/medium data, are fast to train and can be explained (importance, SHAP). Published benchmarks on tabular data (e.g. Grinsztajn et al., 2022) show tree ensembles are still very competitive with deep learning.
- Limitations: cannot **extrapolate** beyond the price range seen in training (a brand-new flagship priced above all training phones will be under-predicted); one-hot encoding of many models inflates the feature space; large forests use much memory; a snapshot of prices ages quickly.

**Image / text data**
- Limitations: raw pixels or words have no meaningful single-column features and the information lies in spatial or sequential structure. Trees on raw pixels/tokens perform poorly unless hand-crafted features or embeddings are created first.
- Better suited: convolutional networks and transformers learn those representations automatically and are the standard choice. Tree ensembles are still useful *on top of* extracted features/embeddings (e.g. image embeddings + metadata).

## 5. My conclusions 
- Hybrid RF + ANN achieved the best test performance (MAE Rs. 15,810, R² 0.865), reducing Ridge's error by 22%.
- Random Forest was a close second and a faster, simpler alternative.
- Boosting did not outperform Random Forest with default settings.
- Stacking did not improve MAE because its base models made similar errors and required more training time.
- Main limitations include asking prices, limited Dialog data, missing RAM values and default model settings.
- Future improvements include tuning boosting models, collecting more data and improving phone-specification extraction.


## 6. References
- Breiman, L. (2001). Random Forests. *Machine Learning*, 45, 5-32.
- Chen, T. & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *KDD*.
- Ke, G. et al. (2017). LightGBM: A Highly Efficient Gradient Boosting Decision Tree. *NeurIPS*.
- Prokhorenkova, L. et al. (2018). CatBoost: unbiased boosting with categorical features. *NeurIPS*.
- Wolpert, D. H. (1992). Stacked generalization. *Neural Networks*, 5(2), 241-259.
- Grinsztajn, L., Oyallon, E. & Varoquaux, G. (2022). Why do tree-based models still outperform deep learning on typical tabular data? *NeurIPS Datasets and Benchmarks*.
- scikit-learn, XGBoost, LightGBM and CatBoost documentation.